In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime
from functools import partial
import gc

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.CutMasks.CutMasks import *

from makedf.mcstat import get_MCstat_unc

from analysis_village.cc1pi.var_configs import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/syst/detector"

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load df

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']

# BNB data
# data_tot_pot = data_hdr_df['TOR875'].sum()
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
pot_str = f"{data_tot_pot} $\\times 10^{18}$"
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))


In [ ]:
def proton_BDT_sideband_mask(df, group_levels):
    BDT_proton_df = df[(is_MIP_candidate_mask(df)) & (df.pfp.trk.bdt_proton_score < -2)]
    
    # Count how many pfps per slice
    candidate_counts = BDT_proton_df.groupby(level=group_levels).size()
 
    # Get only slices with at least 2 pfps
    valid_slices = candidate_counts[candidate_counts == 1].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(df.index.droplevel('rec.slc.reco.pfp..index').isin(valid_slices), index=df.index)

    return final_mask
    

In [ ]:
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV_grid_new_cc1pi.df", keys2load, 100)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

mc_evt_df = perform_truth_matching(mc_bnb_evt_df, mc_bnb_nu_df)

In [ ]:
#Load GiBUU dataframe
pot_weight_col = ('slc', 'wgt', '', '', '', '')

keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_GiBUU_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_GIBUU_gen1.df", keys2load, 100, filter_df = False)
#mc_GiBUU_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV_grid_new_cc1pi.df", keys2load, 100, filter_df = False)
#mc_GiBUU_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV_grid_new_cc1pi.df", keys2load, 100, filter_df = False)

mc_GiBUU_evt_df = mc_GiBUU_df['cc1pi']
mc_GiBUU_nu_df = mc_GiBUU_df['nudf']
mc_GiBUU_hdr_df = mc_GiBUU_df['hdr']

#Add weight column
mc_GiBUU_tot_pot = mc_GiBUU_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_GiBUU_tot_pot))
mc_pot_scale = data_tot_pot / mc_GiBUU_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_GiBUU_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_GiBUU_evt_df))

#Do truth matchign
mc_GiBUU_evt_df = perform_truth_matching(mc_GiBUU_evt_df, mc_GiBUU_nu_df)

# Prepare maks

In [ ]:
def proton_BDT_sideband_mask(df, group_levels):
    BDT_proton_df = df[(is_MIP_candidate_mask(df)) & (df.pfp.trk.bdt_proton_score < -2)]
    
    # Count how many pfps per slice
    candidate_counts = BDT_proton_df.groupby(level=group_levels).size()
 
    # Get only slices with at least 2 pfps
    valid_slices = candidate_counts[candidate_counts == 1].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(df.index.droplevel('rec.slc.reco.pfp..index').isin(valid_slices), index=df.index)

    return final_mask

In [ ]:
def shower_cut_mask(df, group_levels):
    is_pandora_primary_mask = (df.pfp.parent_is_primary == True)
    is_shower_mask = (df.pfp.trackScore >= 0) & (df.pfp.trackScore < CTE.max_shower_track_score)
    energy_mask = (df.pfp.shw.bestplane_energy > 0.200) &  (df.pfp.shw.bestplane_energy < 0.500)
    shower_df = df[is_pandora_primary_mask & is_shower_mask & energy_mask] 
        
    # Count how many pfps per slice
    shower_counts = shower_df.groupby(level=group_levels).size()
    
    # Get only slices with at least 2 pfps
    non_valid_slices = shower_counts[(shower_counts > 0)].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(~df.index.droplevel('rec.slc.reco.pfp..index').isin(non_valid_slices), index=df.index)
   
    return final_mask

In [ ]:
group_levels = ['__ntuple', 'entry', 'rec.slc..index']
def build_event_cumulative_masks(evt_df, sideband = "shower"):
    # Define which BDT mask to use
    
    cut_sequence = [
        ("cosmic",      evt_df.slc.cut.obvious_cosmic),
        ("t0",          evt_df.slc.cut.t0),
        ("FV",          evt_df.slc.cut.inside_FV),
        ("nu_score",    evt_df.slc.nu_score > CTE.min_nu_score),
        ("track",       evt_df.slc.cut.track),
        ("chi2",        evt_df.slc.cut.MIP_candidates),
        ("shower",      evt_df.slc.cut.shower),
        ("angle",       evt_df.slc.cut.angle),
        ("proton_BDT",  evt_df.slc.cut.proton_BDT),
        ("containment", evt_df.slc.cut.containment),
        ("michel",      evt_df.slc.cut.michel),
        ("extra_pion",  evt_df.slc.cut.extra_pion),
        #("bad_tracks",  evt_df.slc.cut_var.n_bad_trks == 0),
        ("energy",      evt_df.slc.cut.energy),
    ]
    
    if sideband == "shower": 
        cut_sequence = [
            ("cosmic",      evt_df.slc.cut.obvious_cosmic),
            ("t0",          evt_df.slc.cut.t0),
            ("FV",          evt_df.slc.cut.inside_FV),
            ("nu_score",    evt_df.slc.nu_score > CTE.min_nu_score),
            ("track",       evt_df.slc.cut.track),
            ("chi2",        evt_df.slc.cut.MIP_candidates),
            ("shower",      shower_cut_mask(evt_df, group_levels) == False),
            ("angle",       evt_df.slc.cut.angle),
            #("proton_BDT",  evt_df.slc.cut.proton_BDT_sideband),
            ("containment", evt_df.slc.cut.containment),
            ("michel",      evt_df.slc.cut.michel),
            #("extra_pion",  evt_df.slc.cut.extra_pion),
            #("bad_tracks",  evt_df.slc.cut_var.n_bad_trks == 0),
            #("energy",      evt_df.slc.cut.energy),
            ("energy",      evt_df.slc.cut.obvious_cosmic),
        ]

    elif sideband == "proton": 
        cut_sequence = [
            ("cosmic",      evt_df.slc.cut.obvious_cosmic),
            ("t0",          evt_df.slc.cut.t0),
            ("FV",          evt_df.slc.cut.inside_FV),
            ("nu_score",    evt_df.slc.nu_score > CTE.min_nu_score),
            ("track",       evt_df.slc.cut.track),
            ("chi2",        evt_df.slc.cut.MIP_candidates),
            ("shower",      evt_df.slc.cut.shower),
            ("angle",       evt_df.slc.cut.angle),
            ("proton_BDT",  proton_BDT_sideband_mask(evt_df,group_levels)),
            ("containment", evt_df.slc.cut.containment),
            ("michel",      evt_df.slc.cut.michel),
            #("extra_pion",  evt_df.slc.cut.extra_pion),
            #("bad_tracks",  evt_df.slc.cut_var.n_bad_trks == 0),
            #("energy",      evt_df.slc.cut.energy),
            ("energy",      evt_df.slc.cut.obvious_cosmic),
        ]

            
        

    # 2️⃣ Build cumulative masks
    cumulative_masks = {}
    current_mask = None

    for name, mask in cut_sequence:

        if current_mask is None:
            current_mask = mask.copy()
        else:
            current_mask = current_mask & mask

        cumulative_masks[name] = current_mask.copy()

    return cumulative_masks

In [ ]:
# --- CV ---
#proton sideband

mc_GiBUU_evt_df_sideband = mc_GiBUU_evt_df[build_event_cumulative_masks(mc_GiBUU_evt_df, sideband = "proton")["energy"]]
mc_evt_df_sideband = mc_evt_df[build_event_cumulative_masks(mc_evt_df, sideband = "proton")["energy"]]

mc_GiBUU_evt_df = mc_GiBUU_evt_df[build_event_cumulative_masks(mc_GiBUU_evt_df, sideband = "")["energy"]]
mc_evt_df = mc_evt_df[build_event_cumulative_masks(mc_evt_df, sideband = "")["energy"]]

# Get all the syst uncert

In [ ]:
from analysis_village.cc1pi.HelperFunctions import HelperFunctions

HelperFunctions.print_purity(mc_evt_df, ('truth','nu_categ','','','',''))
HelperFunctions.print_purity(mc_evt_df_sideband, ('truth','nu_categ','','','',''))
HelperFunctions.print_purity(mc_GiBUU_evt_df, ('truth','nu_categ','','','',''))
HelperFunctions.print_purity(mc_GiBUU_evt_df_sideband, ('truth','nu_categ','','','',''))

In [ ]:
def overlay_hists(mc_df=None,
                  data_df=None,
                  var_config=""): 
    
    if mc_df is not None:
        vardf, _        = get_clipped_evts(mc_df, var_config.var_evt_reco_col, var_config.bins)

        cuts = [
                mc_df.truth.nu_categ == topo
                for topo in topology_list
            ]
        
        var_categ = [vardf[i] for i in cuts]
        weights_categ = [mc_df.loc[cut, ('slc','wgt','','','','')]
            for cut in cuts
        ]
        
        # MC stat err
        each_mc_hist_data = []
        for v, w in zip(var_categ, weights_categ):
            hist_vals, _ = np.histogram(v, weights=w, bins=var_config.bins)
            each_mc_hist_data.append(hist_vals)
        total_mc = np.sum(each_mc_hist_data, axis=0)
        total_mc_bkgd = sum(each_mc_hist_data[1:])

    else:
        vardf = None
        var_categ = None
        total_mc = None
        print("No MC data provided")
        
    # Data
    if data_df is not None:
        vardf_data, _   = get_clipped_evts(data_df, var_config.var_evt_reco_col, var_config.bins)
        total_data, _ = np.histogram(vardf_data, bins=var_config.bins, weights=data_df.slc.wgt)

    return {"cuts": cuts, 
            "total_mc": total_mc, 
            "total_mc_bkgd": total_mc_bkgd,
            "total_data": total_data}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

unisim_keys = []
paired_syst = {} 
file_dir = "/exp/sbnd/data/users/lpelegri/syst/CCBC_rates"
os.makedirs(file_dir, exist_ok=True)

if 'syst_dict' not in locals():
    syst_dict = {}

var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]

univ_hist_GiBUU = {"Ps": {}, "Bs": {}, "nc": {}}
# --- 3. Main Variable Loop ---
for var_config in var_configs:
    var_name = var_config.var_save_name

    ret_GiBUU = overlay_hists(mc_df=mc_GiBUU_evt_df,
                            data_df=mc_GiBUU_evt_df_sideband,
                            var_config=var_config)
    print(ret_GiBUU)
    univ_hist_GiBUU["Ps"][var_name] = np.array(ret_GiBUU["total_mc"] - ret_GiBUU["total_mc_bkgd"])
    univ_hist_GiBUU["Bs"][var_name] = np.array(ret_GiBUU["total_mc_bkgd"])
    univ_hist_GiBUU["nc"][var_name] = np.array(ret_GiBUU["total_data"])

# --- CRITICAL PART FOR SAVING ---
# np.savez does not like deep nesting. To preserve your dict structure:
save_path = os.path.join(file_dir, "GiBUU_univ_hists.npz")
np.savez(save_path, **{k: np.array(v, dtype=object) for k, v in univ_hist_GiBUU.items()})